đánh giá 7 nhóm câu hỏi

metrics: intent accuracy / source relevance / keyword presence / latency



In [ ]:
import os, sys, time

ROOT = os.path.dirname(os.path.abspath(""))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

SEP  = "=" * 70
SEP2 = "-" * 50

# ground truth
TEST_CASES = [
    {
        "group": 1, "label": "Tra cứu dinh dưỡng",
        "query": "100g ức gà có bao nhiêu protein?",
        "expected_intent": "NUTRITION_LOOKUP",
        "expected_source_keywords": ["USDA"],
        "expected_answer_keywords": ["protein", "14"],
        "keyword_match": "all",
    },
    {
        "group": 2, "label": "Bệnh lý + chế độ ăn",
        "query": "Bị tiểu đường nên ăn gì và kiêng gì?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "tiểu đường", "đường huyết", "đái tháo"],
        "expected_answer_keywords": ["tiểu đường", "chất xơ", "hạn chế", "rau", "đường huyết"],
        "keyword_match": "any",
    },
    {
        "group": 3, "label": "Triệu chứng",
        "query": "Bị đau đầu chóng mặt mệt mỏi nên bổ sung gì?",
        "expected_intent": "HEALTH_ADVICE",
        "expected_source_keywords": ["dinh dưỡng", "Vinmec", "thiếu máu"],
        "expected_answer_keywords": ["sắt", "magiê", "vitamin", "thiếu"],
        "keyword_match": "any",
    },
    {
        "group": 4, "label": "Mục tiêu cơ thể (gym/giảm cân)",
        "query": "Sau khi tập gym nên ăn gì để tăng cơ?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "thể thao", "gym", "cơ"],
        "expected_answer_keywords": ["protein", "cơ", "thịt", "trứng", "sữa"],
        "keyword_match": "any",
    },
    {
        "group": 5, "label": "Theo đối tượng (bà bầu)",
        "query": "Bà bầu 3 tháng đầu nên bổ sung gì?",
        "expected_intent": "HEALTH_ADVICE",
        "expected_source_keywords": ["Vinmec", "thai", "bầu", "mang thai"],
        "expected_answer_keywords": ["acid folic", "folate", "sắt", "canxi", "rau", "vitamin"],
        "keyword_match": "any",
    },
    {
        "group": 6, "label": "Ăn chay / ăn kiêng",
        "query": "Ăn chay có đủ protein không, bổ sung từ đâu?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "chay", "protein", "thực vật"],
        "expected_answer_keywords": ["protein", "đậu", "hạt", "đủ"],
        "keyword_match": "any",
    },
    {
        "group": 7, "label": "Câu phức hợp (cả 2 nhánh)",
        "query": "Người bị tiểu đường muốn giảm cân nên ăn gì?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "tiểu đường", "đường huyết"],
        "expected_answer_keywords": ["tiểu đường", "giảm cân", "chất xơ", "rau", "hạn chế"],
        "keyword_match": "any",
    },
]

_results_log = []

def _check_intent(tc, result):
    return result.get("query_type", "") == tc["expected_intent"]

def _check_source(tc, result):
    sources_str = " ".join(result.get("sources", [])).lower()
    return any(kw.lower() in sources_str for kw in tc["expected_source_keywords"])

def _check_keywords(tc, result):
    answer_lower = result.get("answer", "").lower()
    kws = tc["expected_answer_keywords"]
    if tc.get("keyword_match") == "all":
        return all(kw.lower() in answer_lower for kw in kws)
    return any(kw.lower() in answer_lower for kw in kws)

def run_group(tc):
    t0 = time.time()
    result = pipeline.answer(tc["query"])
    latency = time.time() - t0

    ok_intent   = _check_intent(tc, result)
    ok_source   = _check_source(tc, result)
    ok_keywords = _check_keywords(tc, result)

    _results_log.append({
        "group": tc["group"], "latency": latency,
        "ok_intent": ok_intent, "ok_source": ok_source,
        "ok_keywords": ok_keywords,
        "has_answer": bool(result.get("answer", "").strip()),
    })

    print(SEP)
    print(f"NHÓM {tc['group']} — {tc['label']}")
    print(f"Câu hỏi : {tc['query']}")
    print(SEP2)

    actual_intent = result.get("query_type", "?")
    print(f"Intent   : {actual_intent}  [{'đúng' if ok_intent else 'sai'} expected: {tc['expected_intent']}]")

    ents = result.get("entities", {})
    if ents:
        for etype, vals in ents.items():
            if vals:
                print(f"  {etype:10s}: {', '.join(vals)}")
    else:
        print("  (no entity — keyword fallback)")

    nd = result.get("nutrition_data")
    if nd:
        print(f"\nUSDA     : {nd.get('food_description')} | "
              f"{nd.get('nutrient_name')} = "
              f"{nd.get('amount_per_100g')} {nd.get('unit')}/100g")

    sources = result.get("sources", [])
    print(f"\nNguồn    : {', '.join(sources[:3]) if sources else '(none)'}  [{'đúng' if ok_source else 'sai'}]")

    print(f"\nTrả lời  :\n{result.get('answer', '')}")
    print(f"\nUsed LLM : {result.get('used_llm', False)}")
    print(f"Latency  : {latency:.1f}s")
    print(f"Keywords : [{'đúng' if ok_keywords else 'sai'}] '{', '.join(tc['expected_answer_keywords'])}'")
    print(SEP)

print("setup done")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from src.pipeline import NutritionPipeline
pipeline = NutritionPipeline()
print("pipeline ready")

In [ ]:
# Cell 3 — Nhóm 1: Tra cứu dinh dưỡng
run_group(TEST_CASES[0])

In [ ]:
# Cell 4 — Nhóm 2: Bệnh lý + chế độ ăn
run_group(TEST_CASES[1])

In [ ]:
# Cell 5 — Nhóm 3: Triệu chứng
run_group(TEST_CASES[2])

In [ ]:
# Cell 6 — Nhóm 4: Mục tiêu cơ thể (gym)
run_group(TEST_CASES[3])

In [ ]:
# Cell 7 — Nhóm 5: Theo đối tượng (bà bầu)
run_group(TEST_CASES[4])

In [ ]:
# Cell 8 — Nhóm 6: Ăn chay / ăn kiêng
run_group(TEST_CASES[5])

In [ ]:
# Cell 9 — Nhóm 7: Câu phức hợp (cả 2 nhánh)
run_group(TEST_CASES[6])

In [ ]:
n = len(_results_log)
if n == 0:
    print("chưa chạy nhóm nào.")
else:
    ok_intent   = sum(r["ok_intent"]   for r in _results_log)
    ok_source   = sum(r["ok_source"]   for r in _results_log)
    ok_keywords = sum(r["ok_keywords"] for r in _results_log)
    has_answer  = sum(r["has_answer"]  for r in _results_log)
    lats        = [r["latency"] for r in _results_log]

    print("=" * 50)
    print("SUMMARY REPORT")
    print("-" * 50)
    print(f"  Nhóm đã chạy         : {n}/7")
    print(f"  Answer (non-empty)   : {has_answer}/{n}  ({has_answer/n*100:.1f}%)")
    print(f"  Intent accuracy      : {ok_intent}/{n}  ({ok_intent/n*100:.1f}%)")
    print(f"  Source relevance     : {ok_source}/{n}  ({ok_source/n*100:.1f}%)")
    print(f"  Keyword presence     : {ok_keywords}/{n}  ({ok_keywords/n*100:.1f}%)")
    print(f"  Latency avg          : {sum(lats)/n:.1f}s  |  min {min(lats):.1f}s  |  max {max(lats):.1f}s")
    print("=" * 50)

    print("\nChi tiết:")
    print(f"  {'Nhóm':<8} {'Intent':>8} {'Source':>8} {'Keywords':>10} {'Latency':>10}")
    print("  " + "-" * 46)
    for r in _results_log:
        print(f"  {r['group']:<8} "
              f"{'đúng' if r['ok_intent'] else 'sai':>8} "
              f"{'đúng' if r['ok_source'] else 'sai':>8} "
              f"{'đúng' if r['ok_keywords'] else 'sai':>10} "
              f"{r['latency']:>9.1f}s")